# 🌌 APEX Runtime

Welcome to the APEX Runtime. This notebook is your primary workflow interface. Execute cells sequentially to configure, initialize, and control the runtime. The dashboard will launch below and monitor your state live.

---
## 1. Environment

Run this cell to fetch the latest APEX codebase, mount Google Drive, and configure your system path.

In [ ]:
import sys
import subprocess
from pathlib import Path

# 1. Environment Detection
is_colab = "google.colab" in sys.modules
if is_colab:
    print("\n[INFO] Google Colab detected.")
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        print("[SUCCESS] Drive Mounted.")
    except Exception:
        print("[WARNING] Drive mount failed. Using temporary storage.")

# 2. Fetch Codebase
target_dir = Path("/content/APEX") if is_colab else Path.cwd().parent / "APEX"
if not (target_dir / ".git").exists():
    print("\n[INFO] Fetching APEX Runtime...")
    target_dir.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "clone", "-b", "main", "https://github.com/Nikhil-Kumar-Shah/APEX.git", str(target_dir)], check=True)
else:
    print("\n[INFO] Updating APEX Runtime...")
    subprocess.run(["git", "-C", str(target_dir), "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", str(target_dir), "reset", "--hard", "origin/main"], check=True)
    subprocess.run(["git", "-C", str(target_dir), "clean", "-fd"], check=True)

if str(target_dir.resolve()) not in sys.path:
    sys.path.insert(0, str(target_dir.resolve()))

# Clear module cache if restarting
for mod in list(sys.modules.keys()):
    if mod.startswith("runtime"):
        del sys.modules[mod]

print("\n[SUCCESS] Environment Ready.")

---
## 2. Configuration

Edit runtime settings directly below.

In [ ]:
MODEL_ID = "Qwen/Qwen2.5-0.5B"
CACHE_ENABLED = True
PRECISION = "float16"

API_ENABLED = True
AUTHENTICATION = False
ENABLE_TUNNEL = False
TUNNEL_PROVIDER = "cloudflare"
API_PORT = 8000

---
## 3. Initialize Runtime

Run this cell to start the lifecycle, boot the backend managers, and launch the Live Status Monitor.

In [ ]:
from runtime.core.lifecycle import RuntimeLifecycle
from runtime.model.manager import ModelManager
from runtime.core.health import HealthMonitor
from runtime.memory.workspace import WorkspaceManager
from runtime.ui.dashboard import RuntimeDashboard
from runtime.orchestrator.orchestrator import RuntimeOrchestrator
from runtime.api.configuration import APIConfig
from runtime.api.state import RuntimeState
from runtime.api.manager import APIManager

print("Booting APEX Lifecycle...")
lifecycle = RuntimeLifecycle(workspace_path=target_dir)
lifecycle.startup(interactive=False)

cache_path = lifecycle.drive_manager.get_path("cache")
workspaces_dir = lifecycle.drive_manager.persistence_root / "workspaces"

# Initialize Managers
model_manager = ModelManager(cache_path)
health_monitor = HealthMonitor(cache_path, model_manager)
workspace_manager = WorkspaceManager(workspaces_dir)
orchestrator = RuntimeOrchestrator(model_manager, workspace_manager)

workspace_manager.create_workspace("default", "Default Workspace")
workspace_manager.load_workspace("default")

# Setup API Subsystem
api_config = APIConfig(
    api_enabled=API_ENABLED,
    port=API_PORT,
    enable_auth=AUTHENTICATION,
    enable_tunnel=ENABLE_TUNNEL,
    tunnel_provider=TUNNEL_PROVIDER
)
api_state = RuntimeState()
api_manager = APIManager(api_config, api_state, model_manager)

# Launch Dashboard Monitor
dashboard = RuntimeDashboard()
dashboard.api_manager = api_manager
dashboard.bind_managers(lifecycle.config_manager, model_manager, health_monitor, lifecycle, orchestrator)
dashboard.render()
dashboard.refresh()

---
## 4. Download Model

Run to execute standard Hugging Face snapshot downloads. Progress is streamed natively.

In [ ]:
orchestrator.state_machine.transition_to("DOWNLOADING_MODEL")
dashboard.refresh()

model_manager.download_model(MODEL_ID)

orchestrator.state_machine.transition_to("READY")
dashboard.refresh()

---
## 5. Load Model

Run to allocate the model into VRAM.

In [ ]:
orchestrator.state_machine.transition_to("LOADING_MODEL")
dashboard.refresh()

model_manager.load_model(MODEL_ID, precision=PRECISION)
api_state.model_loaded = MODEL_ID

orchestrator.state_machine.transition_to("READY")
dashboard.refresh()

---
## 6. Start API

Run to start the background inference server.

In [ ]:
api_manager.start()
dashboard.refresh()

---
## 7. Shutdown

Run to unload weights and shutdown orchestrator.

In [ ]:
api_manager.stop()
model_manager.unload_model()
orchestrator.shutdown()
dashboard.refresh()
print("Shutdown Complete.")